# Libraries Used

In [1]:
import evaluate
import pandas as pd
import transformers
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers.trainer_utils import EvalPrediction


W0909 18:37:52.364000 15504 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


## Setup...

In [2]:
import sys

is_colab = "google.colab" in sys.modules
if is_colab:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    OUT_DIR = "./drive/MyDrive/Colab Notebooks/ModernBert"
else:
    OUT_DIR = "./ModernBert"

# instantiate the model
NAME = "answerdotai/ModernBERT-base"

In [ ]:
def get_model():
    return transformers.AutoModelForSequenceClassification.from_pretrained(
        NAME,
        dtype="auto",
        num_labels=2,
    )


# set up the dataset and dataset loading
tokenizer = transformers.AutoTokenizer.from_pretrained(NAME)
ds_raw = load_dataset("stanfordnlp/sst2")

# tokenize the dataset entries
ds = ds_raw.map(
    lambda item: tokenizer(
        item["sentence"],
        truncation=True,
        max_length=512,
    )
)

# batch collator
collator = transformers.DataCollatorWithPadding(tokenizer)


def get_trainer(model, type: str):
    training_args = transformers.TrainingArguments(
        output_dir=f"{OUT_DIR}/{type}",
        num_train_epochs=4,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        bf16=True,
        learning_rate=2e-5,
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=1000,
        load_best_model_at_end=True,
    )

    return transformers.Trainer(
        model,
        args=training_args,
        train_dataset=ds["train"],
        eval_dataset=ds["validation"],
        compute_metrics=calc_accuracy,
        processing_class=tokenizer,
        data_collator=collator,
    )

## Regular FT

### Training

In [4]:
model = get_model()

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# set up training params
for param in model.base_model.parameters():
    param.requires_grad_(False)

# accuracy setup
accuracy = evaluate.load("accuracy")


def calc_accuracy(p: EvalPrediction) -> dict:
    return accuracy.compute(
        predictions=p.predictions.argmax(axis=-1),  # type: ignore
        references=p.label_ids,
    )

trainer = get_trainer(model, "sft")

In [6]:
trainer.train(resume_from_checkpoint=True)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
[transformers] Could not locate the best model at ./ModernBert\checkpoint-8420\pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


Step,Training Loss,Validation Loss


TrainOutput(global_step=8420, training_loss=0.0, metrics={'train_runtime': 0.0014, 'train_samples_per_second': 93136228.189, 'train_steps_per_second': 5821964.998, 'total_flos': 1757352769543308.0, 'train_loss': 0.0, 'epoch': 2.0})

### Results

In [7]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy
3.731248,0.444930,8420,0.805046


{'eval_loss': 0.44492968916893005, 'eval_accuracy': 0.805045871559633}

## LoRA

Let's see how many params the traditioal model had...

In [8]:
trainer.get_num_trainable_parameters()

592130

Now let's see what we can get with LoRA

In [9]:
lora_config = LoraConfig(
    r=8,
    task_type=TaskType.SEQ_CLS,
    target_modules=['Wqkv', 'classifier', 'head.dense']
)
lora_model = get_peft_model(get_model(), lora_config)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
lora_model.get_nb_trainable_parameters()[0]

554498

So we have 50000 fewer parameters - about in the same ballpark.

Now let's train...

In [11]:
lora_trainer = get_trainer(lora_model, 'lora')

In [13]:
lora_trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss


KeyboardInterrupt: 